# 🧼 04 - Basic Cleaning: Missing Values, Types, and Duplicates

This notebook is self-contained — it builds its own messy sample data, so there's nothing to download first. You'll clean a small `df` step by step, then put everything together in a full cleaning **pipeline** on a messier sales dataset.

## ✅ Learning Goals
- Detect and count missing values, and know which "missing" values pandas *can't* see (`""`, `'N/A'`, `-1`)
- Inspect a column's values and types **before** cleaning it
- Fill or drop missing data, and choose between them for a good reason
- Convert columns to numbers, dates, and nullable types safely, and find what failed to convert
- Standardize messy text categories
- Find, inspect, and remove duplicate rows
- Build a cleaning pipeline, and validate the result

> **The core habit:** standardize missing values first, inspect before converting, and validate after every major cleaning step.

## 1. Load a Messy Dataset

In [1]:
import pandas as pd
import numpy as np

data = {
    'Name': ['Alice', 'Bob', 'Charlie', 'David', None],
    'Age': ['25', 'thirty', 35, np.nan, '40'],
    'Signup Date': ['2022-01-01', 'not a date', '2022/03/01', None, 'April 5, 2022'],
    'Score': [95.5, None, 88.0, 92.5, ''],
}

df = pd.DataFrame(data)
df

,Name,Age,Signup Date,Score
0,Alice,25,2022-01-01,95.5
1,Bob,thirty,not a date,None
2,Charlie,35,2022/03/01,88.0
3,David,NaN,NaN,92.5
4,NaN,40,"April 5, 2022",


## 2. Detecting Missing or Broken Values

In [2]:
df.isnull()

,Name,Age,Signup Date,Score
0,False,False,False,False
1,False,False,False,True
2,False,False,False,False
3,False,True,True,False
4,True,False,False,False


In [3]:
df.isnull().sum()

Name           1
Age            1
Signup Date    1
Score          1
dtype: int64

💡 `.isna()` is exactly the same as `.isnull()`. You'll see both in other people's code.

### What Counts as Missing?
Not every "missing" value looks missing to pandas:

| Value | What it is | Does `.isna()` catch it? |
|---|---|---|
| `np.nan` / `NaN` | Numeric missing value | ✅ Yes |
| `None` | Python's "nothing" | ✅ Yes |
| `pd.NA` | Pandas' missing-value marker | ✅ Yes |
| `""` | An empty string | ❌ No |
| `'N/A'`, `'Unknown'` | Text placeholders | ❌ No, unless you replace them |
| `-1`, `999` | Number placeholders | ❌ No, unless you replace them |

In [4]:
pd.Series([np.nan, None, pd.NA, '', 'N/A', -1]).isna()

0     True
1     True
2     True
3    False
4    False
5    False
dtype: bool

Pandas can't know that `-1` means "unknown" in **your** dataset, so the missing-value count above is only the *visible* missing data. Section 8 turns the hidden kind into real missing values.

In [5]:
df[df.isnull().any(axis=1)]

,Name,Age,Signup Date,Score
1,Bob,thirty,not a date,None
3,David,NaN,NaN,92.5
4,NaN,40,"April 5, 2022",


Two more counts are handy: the **total** number of missing cells, and the rows where **every** value is missing (often junk rows left over at the end of a file):

In [6]:
print('Total missing cells:', df.isnull().sum().sum())
df[df.isnull().all(axis=1)]          # no completely empty rows here

Total missing cells: 4


,Name,Age,Signup Date,Score


### Inspect Before You Clean
Before changing anything, **look** at what's actually in a column. `value_counts(dropna=False)` shows every distinct value, including the missing ones. `.map(type)` shows what **kind** of value each cell holds:

In [7]:
df['Score'].value_counts(dropna=False)

Score
95.5    1
None    1
88.0    1
92.5    1
        1
Name: count, dtype: int64

In [8]:
df['Age'].map(type).value_counts()

Age
<class 'str'>      3
<class 'int'>      1
<class 'float'>    1
Name: count, dtype: int64

`Age` is a mix of strings (`'25'`, `'thirty'`), an `int`, and a `float` (the `NaN`). A column that holds several types is a sure sign it needs cleaning. `Score` hides an empty string `''` that `.isnull()` didn't count.

## 3. Cleaning Strategy Options

In [9]:
df.fillna({
    'Name': 'Unknown',
    'Age': -1,
    'Signup Date': '1970-01-01',
    'Score': 0.0,
})

,Name,Age,Signup Date,Score
0,Alice,25,2022-01-01,95.5
1,Bob,thirty,not a date,0.0
2,Charlie,35,2022/03/01,88.0
3,David,-1,1970-01-01,92.5
4,Unknown,40,"April 5, 2022",


In [10]:
df.dropna()

,Name,Age,Signup Date,Score
0,Alice,25,2022-01-01,95.5
2,Charlie,35,2022/03/01,88.0


⚠️ **Which is right depends on *why* the value is missing.** Filling every gap with `0` is easy, but a missing test score isn't the same as a score of zero, and it would drag the average down. Ask: is the value unknown, not applicable, or a data-entry error? Then fill it with something that makes sense (like the median), or drop the row.

### Fill or Drop? Some Rules of Thumb

| Situation | Often reasonable |
|---|---|
| The row's identifier (ID, name) is missing | Drop the row, or investigate the source |
| An optional field (a note, a comment) is missing | Keep the row and leave it missing |
| A category is missing, but the row is still useful | Fill with `'Unknown'` or `'Not Provided'` |
| A number is missing, and missing really means zero | Fill with `0` |
| A number is missing, and missing means *unknown* | Leave it, or fill with the median, and say so |
| Most of a column is missing | Consider dropping the column |
| *Whether* it was missing matters | Add a "was missing" flag column first |

You don't have to drop a row for **any** missing value. `subset=` drops only when **important** columns are missing, and `thresh=` keeps columns that have at least that many real values:

In [11]:
print(df.dropna(subset=['Name']).shape)            # drop only rows with no Name
print(df.dropna(axis='columns', thresh=4).columns.tolist())   # keep columns with 4+ real values (all four qualify here)

(4, 4)
['Name', 'Age', 'Signup Date', 'Score']


### Keep a Record of What Was Missing
Filling hides the fact that a value was ever missing. If that might matter later, save a flag **first**:

In [12]:
flags = df.copy()
flags['name_was_missing'] = flags['Name'].isnull()
flags['Name'] = flags['Name'].fillna('Not Provided')
flags[['Name', 'name_was_missing']]

,Name,name_was_missing
0,Alice,False
1,Bob,False
2,Charlie,False
3,David,False
4,Not Provided,True


### Filling From Neighbors: `ffill()` and `bfill()`
When rows are **in order**, like hourly readings, a gap is often best filled from the row before it (`ffill`, forward fill) or after it (`bfill`, backward fill). Don't use these when row order doesn't mean anything.

In [13]:
temps = pd.Series([71.0, None, None, 74.5, None, 73.0],
                  index=['1pm', '2pm', '3pm', '4pm', '5pm', '6pm'])
pd.DataFrame({'raw': temps, 'ffill': temps.ffill(), 'bfill': temps.bfill()})

,raw,ffill,bfill
1pm,71.0,71.0,71.0
2pm,NaN,71.0,74.5
3pm,NaN,71.0,74.5
4pm,74.5,74.5,74.5
5pm,NaN,74.5,73.0
6pm,73.0,73.0,73.0


## 4. Data Type Fixes

In [14]:
df.dtypes

Name              str
Age            object
Signup Date       str
Score          object
dtype: object

Before converting, keep a copy of the raw values, so you can see afterwards exactly which ones **failed**.

⚠️ **The mixed-date trap.** Since pandas 2.0, `pd.to_datetime` guesses the format from the **first** value and turns everything that doesn't match it into `NaT`, **silently**, with no error and no warning. `Signup Date` mixes three formats, so tell pandas with `format='mixed'`:

In [15]:
dates = df['Signup Date']
pd.DataFrame({
    'raw': dates,
    'guessed format': pd.to_datetime(dates, errors='coerce'),
    "format='mixed'": pd.to_datetime(dates, errors='coerce', format='mixed'),
})

,raw,guessed format,format='mixed'
0,2022-01-01,2022-01-01,2022-01-01
1,not a date,NaT,NaT
2,2022/03/01,NaT,2022-03-01
3,NaN,NaT,NaT
4,"April 5, 2022",NaT,2022-04-05


Without `format='mixed'`, two perfectly good dates are lost. When **all** values share one known format, it's even better to state it exactly. For example, `format='%Y-%m-%d'` for `2026-01-05`, or `format='%m/%d/%Y'` for `01/05/2026` (`%Y` = 4-digit year, `%m` = month, `%d` = day).

In [16]:
raw = df.copy()   # the original values, to check what failed

df['Age'] = pd.to_numeric(df['Age'], errors='coerce')
df['Score'] = pd.to_numeric(df['Score'], errors='coerce')
df['Signup Date'] = pd.to_datetime(df['Signup Date'], errors='coerce', format='mixed')
df.dtypes

Name                      str
Age                   float64
Signup Date    datetime64[us]
Score                 float64
dtype: object

`errors='coerce'` turns anything it can't convert into `NaN` (or `NaT` for dates) instead of stopping with an error. Now find the values that **failed**: they're missing now, but weren't before:

In [17]:
for col in ['Age', 'Score', 'Signup Date']:
    failed = raw[col][df[col].isna() & raw[col].notna()]
    print(f'{col:12} failed to convert: {failed.tolist()}')

Age          failed to convert: ['thirty']
Score        failed to convert: ['']
Signup Date  failed to convert: ['not a date']


`'thirty'`, `''`, and `'not a date'` are exactly the values a person would need to look at.

### Type Mismatches
A **type mismatch** is a value stored as the wrong type for what it means:

| Meant to be | Stored as | Problem |
|---|---|---|
| A number | `'19.99'` (text) | Can't be summed or averaged |
| A date | `'01/08/2026'` (text) | Sorts and filters as text, not by date |
| True/False | `'yes'`, `'Y'`, `'no'` | Several spellings of one idea |
| A category | `'IL'`, `'il'`, `'IL '` | One real category counted as three |
| Missing | `''`, `'N/A'`, `-1` | Fake values distort counts and averages |

### Nullable Types and `.astype()`
`.astype()` converts a column that's **already clean** to another type. One catch: a regular integer column (`int64`) **can't hold missing values**, so `.astype('int64')` fails if any are present. Pandas' **nullable** integer type, `'Int64'` (capital **I**), can:

In [18]:
counts = pd.Series([2, None, 1, 3])
print(counts.dtype)                # float64: the None forced decimals
print(counts.astype('Int64'))      # whole numbers again, with <NA> for missing

float64
0       2
1    <NA>
2       1
3       3
dtype: Int64


| Type | Holds missing values? |
|---|---|
| `int64` | ❌ No, so `.astype('int64')` fails with `IntCastingNaNError` |
| `Int64` | ✅ Yes, as `<NA>` |
| `float64` / `Float64` | ✅ Yes |
| `'category'` | ✅ Yes, and saves memory for repeated text (Foundations notebook 08) |

## 5. Impute (Fill In) the Now-Numeric Missing Values

In [19]:
df['Age'] = df['Age'].fillna(df['Age'].median())
df['Score'] = df['Score'].fillna(df['Score'].mean())
df['Signup Date'] = df['Signup Date'].fillna(df['Signup Date'].min())
df['Name'] = df['Name'].fillna('Unknown')

## 6. Cleaned Data Review

In [20]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   Name         5 non-null      str           
 1   Age          5 non-null      float64       
 2   Signup Date  5 non-null      datetime64[us]
 3   Score        5 non-null      float64       
dtypes: datetime64[us](1), float64(2), str(1)
memory usage: 292.0 bytes


In [21]:
df.describe(include='all')

,Name,Age,Signup Date,Score
count,5,5.000000,5,5.00000
unique,5,NaN,NaN,NaN
top,Alice,NaN,NaN,NaN
freq,1,NaN,NaN,NaN
mean,NaN,34.000000,2022-01-31 14:24:00,92.00000
min,NaN,25.000000,2022-01-01 00:00:00,88.00000
25%,NaN,35.000000,2022-01-01 00:00:00,92.00000
50%,NaN,35.000000,2022-01-01 00:00:00,92.00000
75%,NaN,35.000000,2022-03-01 00:00:00,92.50000
max,NaN,40.000000,2022-04-05 00:00:00,95.50000


## 7. Duplicate Rows
The same record entered twice is a common problem, and it quietly inflates counts and totals.

In [22]:
orders = pd.DataFrame({
    'order_id': [101, 102, 102, 103, 104, 104],
    'customer': ['Ana', 'Ben', 'Ben', 'Cy', 'Deb', 'Deb'],
    'amount':   [25.0, 40.0, 40.0, 15.5, 60.0, 65.0],
})

print(orders.duplicated())                 # True for each repeat of an earlier row
print('Exact duplicates:', orders.duplicated().sum())

0    False
1    False
2     True
3    False
4    False
5    False
dtype: bool
Exact duplicates: 1


In [23]:
orders.drop_duplicates()

,order_id,customer,amount
0,101,Ana,25.0
1,102,Ben,40.0
3,103,Cy,15.5
4,104,Deb,60.0
5,104,Deb,65.0


Order 104 appears twice with **different** amounts, so it isn't an exact duplicate. To treat rows as duplicates when only *some* columns match, use `subset=`. Use `keep='last'` to keep the most recent copy instead of the first:

In [24]:
orders.drop_duplicates(subset=['order_id'], keep='last')

,order_id,customer,amount
0,101,Ana,25.0
2,102,Ben,40.0
3,103,Cy,15.5
5,104,Deb,65.0


`orders.duplicated()` only marks the **second** copy. To see **every** member of a duplicate group side by side, use `keep=False`. This is the best way to decide what the duplicates actually are before deleting anything:

In [25]:
orders[orders.duplicated(subset=['order_id'], keep=False)].sort_values('order_id')

,order_id,customer,amount
1,102,Ben,40.0
2,102,Ben,40.0
4,104,Deb,60.0
5,104,Deb,65.0


Order 102 is the same purchase entered twice, so it's safe to drop one copy. Order 104 has two **different** amounts. Is it a correction, a second purchase, or an error? **Don't delete duplicates until you understand what they mean.** When the rows have a date, sort by it first, then `keep='last'` keeps the most recent version.

## 8. Replacing Placeholder Values
Datasets often mark missing data with a **code** instead of leaving it blank, such as `-1`, `999`, or `'N/A'`. Pandas sees those as real values, so they hide from `.isnull()` and distort the math. `.replace()` turns them into real `NaN`s:

In [26]:
survey = pd.DataFrame({
    'age':    [34, -1, 27, 999, 45],
    'answer': ['yes', 'N/A', 'no', 'yes', 'N/A'],
})
print('Mean age before:', survey['age'].mean())

survey = survey.replace({-1: np.nan, 999: np.nan, 'N/A': np.nan})
print('Mean age after: ', survey['age'].mean())
survey

Mean age before: 220.8
Mean age after:  35.333333333333336


,age,answer
0,34.0,yes
1,NaN,NaN
2,27.0,no
3,NaN,yes
4,45.0,NaN


Replacing a value **everywhere** can be too blunt. `-1` might be a placeholder in one column but a real value in another. A **nested** dictionary limits each replacement to one column: `{column: {old: new}}`:

In [27]:
readings = pd.DataFrame({
    'temp_change': [-1, 2, -3],          # -1 is a real value here
    'sensor_id':   [-1, 17, 22],         # ...but means "unknown" here
})
readings.replace({'sensor_id': {-1: np.nan}})

,temp_change,sensor_id
0,-1,NaN
1,2,17.0
2,-3,22.0


💡 If you know a file's placeholders up front, handle them **while loading**: `pd.read_csv('file.csv', na_values=['N/A', 'Unknown', -1])` (Foundations notebook 06). Be careful with `0`, which is usually a **real** value (zero items ordered), not a missing one.

## 9. Cleaning Text Categories
To a computer, `'IL'`, `'il'`, and `'IL '` are three **different** values. Standardize whitespace and capitalization first, then map alternate spellings to one value:

In [28]:
states = pd.Series(['IL', 'il', ' IL ', 'Illinois', 'WI', 'wisconsin', 'WI'])
print(states.value_counts().to_dict())       # 6 "different" states!

clean = states.str.strip().str.upper().replace({'ILLINOIS': 'IL', 'WISCONSIN': 'WI'})
print(clean.value_counts().to_dict())        # really just 2

{'WI': 2, 'IL': 1, 'il': 1, ' IL ': 1, 'Illinois': 1, 'wisconsin': 1}
{'IL': 4, 'WI': 3}


For **yes/no** answers, `.map()` with a dictionary is safer than `.replace()`. Anything the dictionary doesn't cover becomes `NaN`, so unexpected values **show up** instead of slipping through unchanged:

In [29]:
answers = pd.Series([' Yes', 'y', 'NO', 'n ', 'maybe'])
lookup = {'yes': True, 'y': True, 'no': False, 'n': False}

tidy = answers.str.strip().str.lower()
print('replace:', tidy.replace(lookup).tolist())   # 'maybe' quietly survives
print('map:    ', tidy.map(lookup).tolist())       # 'maybe' becomes NaN, so you'll notice

replace: [True, True, False, False, 'maybe']
map:     [True, True, False, False, nan]


## 10. Putting It Together: A Cleaning Pipeline
Here's a messier dataset with every problem from this notebook: `None`, empty strings, text and number placeholders, numbers stored as text, mixed date formats, stray spaces, and a duplicate row.

In [30]:
sales = pd.DataFrame({
    'purchase_id':   [101, 102, 103, 104, 104, 105],
    'customer':      ['Avery', 'Jordan', None, 'Riley', 'Riley', ''],
    'amount':        ['19.99', 'N/A', '-1', '42.50', '42.50', ' 15.00 '],
    'quantity':      [2, None, 1, 3, 3, -1],
    'purchase_date': ['2026-01-05', '01/08/2026', 'not recorded', '2026-01-12', '2026-01-12', ''],
    'state':         ['IL', 'il ', 'Unknown', 'WI', 'WI', 'N/A'],
})
sales

,purchase_id,customer,amount,quantity,purchase_date,state
0,101,Avery,19.99,2.0,2026-01-05,IL
1,102,Jordan,N/A,NaN,01/08/2026,il
2,103,NaN,-1,1.0,not recorded,Unknown
3,104,Riley,42.50,3.0,2026-01-12,WI
4,104,Riley,42.50,3.0,2026-01-12,WI
5,105,,15.00,-1.0,,N/A


A good pipeline works on a **copy**, so the raw data stays untouched for comparison, and it goes in a sensible order: **standardize the missing values first**, then clean text, then convert types, then remove duplicates and unusable rows.

In [31]:
clean_sales = sales.copy()

# 1. Turn placeholders into real missing values, column by column
clean_sales = clean_sales.replace({
    'customer':      {'': pd.NA},
    'amount':        {'N/A': pd.NA, '-1': pd.NA},
    'quantity':      {-1: pd.NA},
    'purchase_date': {'not recorded': pd.NA, '': pd.NA},
    'state':         {'Unknown': pd.NA, 'N/A': pd.NA},
})

# 2. Clean text
clean_sales['state'] = clean_sales['state'].str.strip().str.upper()

# 3. Convert types
clean_sales['amount'] = pd.to_numeric(clean_sales['amount'], errors='coerce')
clean_sales['quantity'] = clean_sales['quantity'].astype('Int64')
clean_sales['purchase_date'] = pd.to_datetime(clean_sales['purchase_date'], errors='coerce', format='mixed')

# 4. Remove exact duplicates, then rows missing the fields a report needs
clean_sales = clean_sales.drop_duplicates()
clean_sales = clean_sales.dropna(subset=['purchase_id', 'amount', 'purchase_date'])

clean_sales

,purchase_id,customer,amount,quantity,purchase_date,state
0,101,Avery,19.99,2,2026-01-05,IL
3,104,Riley,42.50,3,2026-01-12,WI


## 11. Validate After Cleaning
Cleaning isn't done until you've **checked** it. Compare before and after:

In [32]:
print('shape:     ', sales.shape, '→', clean_sales.shape)
print('duplicates:', sales.duplicated().sum(), '→', clean_sales.duplicated().sum())
print('repeated IDs after:', clean_sales['purchase_id'].duplicated().sum())
print()
print(pd.DataFrame({'dtype before': sales.dtypes, 'dtype after': clean_sales.dtypes,
                    'missing before': sales.isna().sum(), 'missing after': clean_sales.isna().sum()}))

shape:      (6, 6) → (2, 6)
duplicates: 1 → 0
repeated IDs after: 0

              dtype before     dtype after  missing before  missing after
purchase_id          int64           int64               0              0
customer               str             str               1              0
amount                 str         float64               0              0
quantity           float64           Int64               1              0
purchase_date          str  datetime64[us]               0              0
state                  str             str               0              0


In [33]:
print(clean_sales['amount'].describe())
print('Dates from', clean_sales['purchase_date'].min().date(), 'to', clean_sales['purchase_date'].max().date())
clean_sales['state'].value_counts(dropna=False)

count     2.000000
mean     31.245000
std      15.916974
min      19.990000
25%      25.617500
50%      31.245000
75%      36.872500
max      42.500000
Name: amount, dtype: float64
Dates from 2026-01-05 to 2026-01-12


state
IL    1
WI    1
Name: count, dtype: int64

Look at the **missing before** column: it found only 2 gaps. The placeholders (`'N/A'`, `-1`, `'Unknown'`, `''`) were invisible to `.isna()` until step 1 turned them into real missing values. That's why the pipeline standardizes missing values **first**.

Then ask yourself:
- Did any **valid** rows disappear? (Here, 104's duplicate, plus 102, 103, and 105, which had no usable amount or date. Is that what the report needs?)
- Did every fake value become a **real** missing value?
- Did the number and date columns convert, and what **failed**?
- Are impossible values still there (negative amounts, future dates)?
- Did duplicate removal use the right rule?

> Cleaning isn't just deleting inconvenient values. It's deciding which values are **valid**, which are **unknown**, and which need **investigating** before you can trust the analysis.

## 🏋️ Practice

### Practice 1 — Count and Compare (easy)
Before running any cleaning steps, how many total missing values were in the *original* `data` dictionary above (count them by hand, then verify with `.isnull().sum().sum()` on a fresh copy)?

In [34]:
original = pd.DataFrame(data)
# Your code here
original.isnull().sum().sum()

np.int64(4)

### Practice 2 — Real Missing Data (medium)
Load `ufo.csv` (in this folder) and print how many missing values each column has. Which column is the messiest?

In [35]:
ufo = pd.read_csv('ufo.csv')
# Your code here
print(ufo.isnull().sum())
# 'country' is the messiest column

date_time                        0
city_area                        0
state                         5797
country                       9670
ufo_shape                     1932
encounter_length                 3
described_encounter_length       0
description                     15
date_documented                  0
latitude                         1
longitude                        0
dtype: int64


### Practice 3 — Fix a Type Column (harder)
In `ufo.csv`, the `date_time` column is stored as text, not an actual date. Convert it with `pd.to_datetime(..., errors='coerce')`, then check how many rows failed to convert (became `NaT`) with `.isnull().sum()`.

In [36]:
# Your code here
ufo['date_time'] = pd.to_datetime(ufo['date_time'], errors='coerce')
print(ufo['date_time'].isnull().sum())

694


### Practice 4 — Duplicate Names (medium)
`students.csv` has no exact duplicate rows, but some students share a **first** name. Use `duplicated(subset=['first_name'])` to list every repeated first name. Then make a version with only one student per first name.

In [37]:
students = pd.read_csv('students.csv')
# Your code here
students[students.duplicated(subset=['first_name'], keep=False)]

,first_name,last_name,math_score,science_score
2,Joshua,Adams,61,100
3,Jeffrey,Zuniga,77,99
7,Christopher,Daniel,66,64
9,Anthony,Gray,65,74
12,Anthony,Hawkins,62,74
16,Jeffrey,Perez,74,89
22,Matthew,Walter,94,73
25,Matthew,Foster,88,98
26,Christopher,Rocha,97,100
44,Christopher,Brown,62,64


In [38]:
students[students.duplicated(subset=['first_name'], keep='last')]

,first_name,last_name,math_score,science_score
2,Joshua,Adams,61,100
3,Jeffrey,Zuniga,77,99
7,Christopher,Daniel,66,64
9,Anthony,Gray,65,74
22,Matthew,Walter,94,73
26,Christopher,Rocha,97,100


### Practice 5 — Standardize and Convert (medium)
1. Standardize this `state` Series so it contains only `'IL'` and `'WI'`, then show its `value_counts()`.
2. Convert `qty` to the nullable `'Int64'` type after turning its `-1` placeholder into a real missing value. Confirm the result holds `<NA>`.

In [39]:
state = pd.Series(['il', 'IL ', 'Illinois', 'WI', ' wi', 'Wisconsin'])
qty = pd.Series([4, -1, 2, 7, -1])
# Your code here
# Standardize this `state` Series so it contains only `'IL'` and `'WI'`, then show its `value_counts()`.
state = state.str.strip().str.upper()
state = state.where(state.isin(['IL', 'WI']), other=pd.NA)
state.value_counts()

IL    2
WI    2
Name: count, dtype: int64

In [40]:
# Convert `qty` to the nullable `'Int64'` type after turning its `-1` placeholder into a real missing value. Confirm the result holds `<NA>`.
qty = qty.replace(-1, pd.NA)
qty = qty.astype('Int64')
print(qty)

0       4
1    <NA>
2       2
3       7
4    <NA>
dtype: Int64


## ✏️ Your Turn
Modify the `data` dictionary at the top of this notebook. Add:
- A new column with some `None` and `""` values
- At least one row with every column filled in incorrectly
- A placeholder value (like `'N/A'` or `-1`) that `.isnull()` won't catch

Then re-run the notebook and fix it step by step. Use `raw` (section 4) to show which values failed to convert, and compare the before and after missing-value counts.

In [41]:
# Your turn here
mod_df = pd.DataFrame(data)
mod_df['new_column'] = ['None', '', 'None', '', 'None']
mod_df.loc[len(mod_df)] = ['john ', 'sixty-seven', 'october 28 2026', -1, 'N/A']

clean_mod_df = mod_df.copy()

# 1. Turn placeholders into real missing values, column by column
clean_mod_df = clean_mod_df.replace({
    'Name': {'None': pd.NA, '': pd.NA},
    'Age': {'': pd.NA},
    'Signup Date': {'N/A': pd.NA},
    'Score': {'': pd.NA, -1: pd.NA},
    'new_column': {'None': pd.NA, '': pd.NA},
})

# 2. Clean text
clean_mod_df['Name'] = clean_mod_df['Name'].str.strip()

# 3. Convert types
clean_mod_df['Age'] = pd.to_numeric(clean_mod_df['Age'], errors='coerce')
clean_mod_df['Signup Date'] = pd.to_datetime(clean_mod_df['Signup Date'], errors='coerce', format='mixed')
clean_mod_df['Score'] = pd.to_numeric(clean_mod_df['Score'], errors='coerce')

# 4. Remove exact duplicates, then rows missing the fields a report needs
clean_mod_df = clean_mod_df.drop_duplicates()
clean_mod_df = clean_mod_df.dropna(subset=['Name', 'Age', 'Signup Date', 'Score'])

clean_mod_df


,Name,Age,Signup Date,Score,new_column
0,Alice,25.0,2022-01-01,95.5,NaN
2,Charlie,35.0,2022-03-01,88.0,NaN


In [42]:
for col in ['Age', 'Score', 'Signup Date']:
    failed = raw[col][df[col].isna() & raw[col].notna()]
    print(f'{col:12} failed to convert: {failed.tolist()}')

print('shape:     ', mod_df.shape, '→', clean_mod_df.shape)
print('duplicates:', mod_df.duplicated().sum(), '→', clean_mod_df.duplicated().sum())
print(pd.DataFrame({'dtype before': mod_df.dtypes, 'dtype after': clean_mod_df.dtypes,
                    'missing before': mod_df.isna().sum(), 'missing after': clean_mod_df.isna().sum()}))

Age          failed to convert: []
Score        failed to convert: []
Signup Date  failed to convert: []
shape:      (6, 5) → (2, 5)
duplicates: 0 → 0
            dtype before     dtype after  missing before  missing after
Name                 str             str               1              0
Age               object         float64               1              0
Signup Date          str  datetime64[us]               1              0
Score             object         float64               1              0
new_column           str             str               0              2


## 🔥 Challenge (Optional) — Clean Real Data End-to-End
Load `ufo.csv` and build a cleaning **pipeline** like section 10:
- Identify which columns have missing values, and inspect the messiest one with `value_counts(dropna=False)`
- Use a mix of `.replace()`, `.fillna()`, `.dropna(subset=...)`, `pd.to_numeric()`, and `pd.to_datetime()` to clean it. Keep the raw data in a separate variable.
- **Validate** like section 11: compare `shape`, `dtypes`, missing-value counts, and duplicate counts before and after, and show which `date_time` values failed to convert. (*Hint:* look at their times.)

In [43]:
# Your code here

---
### 📝 Summary
| Concept        | Tool/Function                          |
|----------------|------------------------------------------|
| Detect nulls   | `df.isnull()`, `df.isnull().sum()`, `.sum().sum()` |
| Inspect a column | `.value_counts(dropna=False)`, `.map(type).value_counts()` |
| Drop rows      | `df.dropna()`, `df.dropna(subset=[...])`  |
| Drop sparse columns | `df.dropna(axis='columns', thresh=n)` |
| Fill values    | `df.fillna()`, `.ffill()`, `.bfill()`     |
| Flag missing   | `df['x_was_missing'] = df['x'].isnull()`  |
| Convert types  | `pd.to_numeric(errors='coerce')`, `pd.to_datetime(errors='coerce', format='mixed')` |
| Nullable integers | `.astype('Int64')`                     |
| Replace values | `df.replace()`, nested `{col: {old: new}}` |
| Clean text     | `.str.strip().str.upper()`, `.map(lookup)` |
| Find duplicates | `df.duplicated()`, `.sum()`, `keep=False` to see all copies |
| Drop duplicates | `df.drop_duplicates(subset=..., keep=...)` |